    # E3 Round1 shard

    This notebook is fully pre-filled for **DGP1**, training
    **n=500**, mode **confirmatory**, and emitted seeds
    **800..899**. Run cells from top to bottom.
    There is no configuration cell to edit. The CSV checkpoint is written to
    `/content` after every completed replication. This notebook assumes the
    session runs to completion; a runtime failure loses the local checkpoint.
    

In [ ]:
import os, platform, subprocess, time

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.stdout.strip()

print("hostname:", platform.node() or "n/a")
print("os:", platform.platform())
print("nproc:", sh("nproc"))
print("cpu:", sh("grep -m1 -E 'model name' /proc/cpuinfo") or "n/a")
print(sh("free -g") or "RAM information unavailable")
print("mc.cores is fixed at 2; model fits are fixed to one thread.")


In [ ]:
import subprocess
p = subprocess.run(
    ["bash", "-lc", "apt-get -qq update >/dev/null && "
     "apt-get -qq install -y --no-install-recommends "
     "r-base r-base-dev libcurl4-openssl-dev >/dev/null 2>&1"],
    capture_output=True, text=True)
if p.returncode != 0:
    print(p.stdout[-1000:])
    print(p.stderr[-2000:])
    raise RuntimeError("R installation failed")
print(subprocess.check_output(["R", "--version"], text=True).splitlines()[0])


In [ ]:
import hashlib, os, re, shutil, subprocess, sys
from pathlib import Path

BUNDLE_FOLDER_URL = 'https://drive.google.com/drive/folders/1w3quuskj25CBOFCGG0mTRGUHcufPpdb3?usp=sharing'
BUNDLE_SHA256 = '12d223bc0fcef624c1ff4cc35c5d7ecc1b1f9b05aa84ecd9d9e4a5a3382bae3c'
assert re.fullmatch(r"[0-9a-fA-F]{64}", BUNDLE_SHA256), \
    "Bundle SHA256 is missing or malformed; regenerate the notebook."

BUNDLE_DOWNLOAD_DIR = Path("/content/tisca_bundle_download")
if BUNDLE_DOWNLOAD_DIR.exists():
    shutil.rmtree(BUNDLE_DOWNLOAD_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gdown"],
    check=True,
)
download = subprocess.run(
    [sys.executable, "-m", "gdown", "--folder", BUNDLE_FOLDER_URL,
     "--output", str(BUNDLE_DOWNLOAD_DIR), "--remaining-ok"],
    capture_output=True, text=True,
)
print(download.stdout[-4000:])
if download.returncode != 0:
    print(download.stderr[-4000:])
    raise RuntimeError("Google Drive bundle download failed")
tar_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.tar.gz"))
sha_candidates = sorted(BUNDLE_DOWNLOAD_DIR.rglob("tisca_rlib.sha256"))
assert len(tar_candidates) == 1, f"expected one tarball, found {tar_candidates}"
assert len(sha_candidates) <= 1, f"expected at most one checksum file, found {sha_candidates}"
if sha_candidates:
    published_sha = sha_candidates[0].read_text().split()[0].lower()
    assert published_sha == BUNDLE_SHA256.lower(), \
        "published tisca_rlib.sha256 differs from the generated notebook"
    print("verified published checksum sidecar:", sha_candidates[0])
else:
    print("no tisca_rlib.sha256 sidecar in the shared folder; "
          "verifying the tarball against the embedded SHA256")
download_path = "/content/_dl_tisca_rlib.tar.gz"
shutil.copy2(tar_candidates[0], download_path)
with open(download_path, "rb") as f:
    observed_sha = hashlib.sha256(f.read()).hexdigest()
assert observed_sha == BUNDLE_SHA256.lower(), "R library bundle SHA mismatch"
if os.path.isdir("/content/tisca_rlib"):
    shutil.rmtree("/content/tisca_rlib")
subprocess.run(["tar", "xzf", download_path, "-C", "/content"], check=True)
LIBDIR = "/content/tisca_rlib/rlib"
assert os.path.isdir(LIBDIR), "bundle did not restore the expected rlib directory"
print("bundle restored:", LIBDIR, "from", BUNDLE_FOLDER_URL)


In [ ]:
import os, urllib.request

RUNCELL_URL = (
    "https://raw.githubusercontent.com/hugogobato/"
    "Test-Informed-Simulation-Count-Algorithm-TISCA/main/"
    "experiments/E3_mvbcf_casestudy/run_cell.R"
)
MVBCF_CPP_URL = (
    "https://raw.githubusercontent.com/Nathan-McJames/MVBCF_Paper/"
    "main/MVBCF_Code.cpp"
)
os.makedirs("/content/e3", exist_ok=True)
urllib.request.urlretrieve(RUNCELL_URL, "/content/e3/run_cell.R")
# The upstream C++ is downloaded at runtime and is never committed here.
urllib.request.urlretrieve(MVBCF_CPP_URL, "/content/e3/MVBCF_Code.cpp")
assert os.path.getsize("/content/e3/run_cell.R") > 1000
assert os.path.getsize("/content/e3/MVBCF_Code.cpp") > 10000
with open("/content/e3/run_cell.R") as f:
    run_cell_source = f.read()
required_fixes = [
    "nthread = nthread_global",
    "general_params = list(num_threads = nthread_global",
    "acquired <- dir.create(lk",
]
missing_fixes = [item for item in required_fixes if item not in run_cell_source]
assert not missing_fixes, (
    "GitHub main is serving a stale run_cell.R. Commit and push the "
    f"corrected driver before running this notebook; missing: {missing_fixes}"
)
print("downloaded run_cell.R and upstream MVBCF_Code.cpp")


In [ ]:
import os, subprocess

compile_script = "\n".join([
    ".libPaths(c('/content/tisca_rlib/rlib', .libPaths()))",
    "if (!requireNamespace('Rcpp', quietly=TRUE) ||",
    "    !requireNamespace('RcppArmadillo', quietly=TRUE) ||",
    "    !requireNamespace('RcppDist', quietly=TRUE)) stop('bundle missing Rcpp dependencies')",
    "library(Rcpp)",
    "sourceCpp('/content/e3/MVBCF_Code.cpp')",
    "stopifnot(is.function(fast_bart))",
    "cat('FAST_BART_OK\\n')",
])
with open("/content/e3/compile.R", "w") as f:
    f.write(compile_script)
p = subprocess.run(["Rscript", "/content/e3/compile.R"],
                   capture_output=True, text=True)
print(p.stdout[-3000:])
if p.returncode != 0 or "FAST_BART_OK" not in p.stdout:
    print(p.stderr[-3000:])
    raise RuntimeError("upstream MVBCF C++ compilation failed")
print("fast_bart() compiled")


    ## Fixed shard configuration

    The constants below were generated from `shard_table.csv`. They are
    assertions, not operator inputs. A dropped session can be restarted by
    uploading this same notebook. Successful seed rows are preserved, while
    failed or malformed checkpoint rows are backed up, removed, and retried.
    

In [ ]:
import csv, os, shutil, subprocess, time

DGP = 1
N = 500
MODE = 'confirmatory'
SHARD_ID = '09'
CLI_SEED_START = 800
CLI_SEED_END = 899
EXPECTED_SEED_START = 800
EXPECTED_SEED_END = 899
MC_CORES = 2
OUTPUT_DIR = "/content/TISCA_E3"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'E3_DGP1_n500_confirmatory_shard09_seeds800-899.csv')
GIT_SHA = "not-a-git-build"

assert DGP in (1, 2, 3)
assert N in (100, 500)
assert MODE in ("pilot", "confirmatory")
assert MC_CORES == 2
assert CLI_SEED_START == 0 or CLI_SEED_START > 0
assert CLI_SEED_END >= CLI_SEED_START
assert EXPECTED_SEED_END - EXPECTED_SEED_START + 1 == CLI_SEED_END - CLI_SEED_START + 1
os.makedirs(OUTPUT_DIR, exist_ok=True)

def emitted_seed(raw_seed):
    return raw_seed + 1000001 if MODE == "pilot" else raw_seed

def contiguous_ranges(values):
    values = sorted(values)
    if not values:
        return []
    out = []
    start = previous = values[0]
    for value in values[1:]:
        if value != previous + 1:
            out.append((start, previous))
            start = value
        previous = value
    out.append((start, previous))
    return out

expected = {emitted_seed(i) for i in range(CLI_SEED_START, CLI_SEED_END + 1)}

def checkpoint_seed(row):
    raw = (row.get("seed") or "").strip()
    try:
        return int(raw)
    except (TypeError, ValueError):
        return None

existing = set()
if os.path.exists(OUTPUT_CSV):
    with open(OUTPUT_CSV, newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if not fieldnames or "seed" not in fieldnames or "converged_flag" not in fieldnames:
        raise RuntimeError("checkpoint CSV is missing required columns")

    kept_rows = []
    rejected = {"malformed": 0, "failed": 0, "duplicate": 0}
    for row in rows:
        seed = checkpoint_seed(row)
        if seed is None or None in row or any(value is None for value in row.values()):
            rejected["malformed"] += 1
            continue
        if seed not in expected:
            raise RuntimeError(
                f"checkpoint contains seed {seed} outside this shard; "
                "check that the correct CSV was uploaded"
            )
        if row.get("converged_flag") != "1":
            rejected["failed"] += 1
            continue
        if seed in existing:
            rejected["duplicate"] += 1
            continue
        existing.add(seed)
        kept_rows.append(row)

    if any(rejected.values()):
        backup = f"{OUTPUT_CSV}.pre_repair_{int(time.time())}.csv"
        shutil.copy2(OUTPUT_CSV, backup)
        temporary = OUTPUT_CSV + ".repair.tmp"
        with open(temporary, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(kept_rows)
        os.replace(temporary, OUTPUT_CSV)
        print("checkpoint repair:", rejected, "backup:", backup)

# A killed older run may leave either the former file lock or the new
# atomic lock directory behind. No R process is active at this point.
stale_lock = OUTPUT_CSV + ".lock"
if os.path.isdir(stale_lock):
    shutil.rmtree(stale_lock)
    print("removed stale checkpoint lock directory")
elif os.path.isfile(stale_lock):
    os.remove(stale_lock)
    print("removed stale checkpoint lock file")

missing_raw = [i for i in range(CLI_SEED_START, CLI_SEED_END + 1)
               if emitted_seed(i) not in existing]
print("checkpoint:", len(existing), "rows; missing:", len(missing_raw))
print("Local checkpoint CSV:", OUTPUT_CSV)


In [ ]:
# Run only missing contiguous ranges. This makes a re-upload idempotent
# while preserving run_cell.R's shard-offset-invariant RNG streams.
env = dict(os.environ)
env["R_LIBS"] = LIBDIR + ":" + env.get("R_LIBS", "")
env["TISCA_MVBCF_CPP"] = "/content/e3/MVBCF_Code.cpp"
env["TISCA_GIT_SHA"] = GIT_SHA
for raw_start, raw_end in contiguous_ranges(missing_raw):
    cmd = ["Rscript", "/content/e3/run_cell.R", str(DGP), str(N),
           str(raw_start), str(raw_end), "--out", OUTPUT_CSV,
           "--cores", str(MC_CORES), "--mode", MODE]
    print("running missing range:", " ".join(cmd))
    t0 = time.time()
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print(result.stdout[-6000:])
    print("range wall-clock: %.1f min" % ((time.time() - t0) / 60.0))
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError("run_cell.R failed")


In [ ]:
import csv, os
with open(OUTPUT_CSV, newline="") as f:
    rows = list(csv.DictReader(f))
malformed = [r.get("seed") for r in rows if checkpoint_seed(r) is None]
assert not malformed, f"checkpoint contains malformed seeds: {{malformed[:5]}}"
got = [checkpoint_seed(r) for r in rows]
expected = list(range(EXPECTED_SEED_START, EXPECTED_SEED_END + 1))
duplicates = len(got) - len(set(got))
missing = sorted(set(expected) - set(got))
unexpected = sorted(set(got) - set(expected))
failures = [r for r in rows if r.get("converged_flag") != "1"]
print("completed rows:", len(rows), "of", len(expected))
print("duplicate seeds:", duplicates, "missing:", len(missing),
      "unexpected:", len(unexpected))
print("converged_flag failures:", len(failures))
print("checkpoint bytes:", os.path.getsize(OUTPUT_CSV))
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    print("Downloaded:", OUTPUT_CSV)
except Exception as e:
    print("(Not on Colab / download skipped):", e)
assert duplicates == 0, "shard checkpoint contains duplicate seeds"
assert not unexpected, f"checkpoint has seeds outside this shard: {{unexpected[:5]}}"
assert not missing, f"shard checkpoint is incomplete; missing seeds: {{missing[:10]}}"
assert not failures, (
    f"{{len(failures)}} replications failed; rerun the notebook to retry them. "
    f"First errors: {{[r.get('error_message') for r in failures[:3]]}}"
)
